In [2]:
import numpy as np
import pandas as pd
import harmonica as hm
import matplotlib.pyplot as plt
import verde as vd
import pygmt

In [5]:
from polartoolkit import fetch, maps, profiles, regions, utils

In [6]:
df = pd.read_csv('Ice_Bridge.csv')

In [7]:
df = df.groupby('line')

blocked_dfs = []
for name, line in df:
    blocked = utils.block_reduce(
        line,
        np.median,
        spacing=500,
        center_coordinates=False,
        input_coord_names=["easting", "northing"],
        input_data_names=line.columns.drop(['line']),
    )
    blocked['line'] = name
    blocked_dfs.append(blocked)

df = pd.concat(blocked_dfs).reset_index(drop=True).sort_values(by=['line'])
df

,easting,northing,Unnamed: 0,height,levelling_correction,free_air_gravity_70s,free_air_gravity_100s,free_air_gravity_140s,line
0,-1297465.500,-1121985.48,9582.0,517.610,-28.61,-28.310,-30.560,2.7310,1
1,-1297185.250,-1122071.97,9578.0,516.400,-28.03,-27.890,-30.340,2.9230,1
2,-1296685.900,-1122197.03,9571.0,517.580,-27.05,-27.210,-29.970,2.7380,1
3,-1296178.230,-1122290.49,9564.0,519.560,-26.14,-26.600,-29.590,2.0110,1
4,-1295665.600,-1122360.89,9557.0,520.080,-25.33,-26.070,-29.200,1.2900,1
...,...,...,...,...,...,...,...,...,...
54449,-1433673.050,-664391.32,284844.0,1267.130,23.32,22.650,27.440,0.1160,813
54450,-1433299.960,-664138.45,284851.0,1272.500,25.38,24.510,28.850,0.1670,813
54451,-1390177.165,-664151.36,285403.5,2452.985,64.59,67.555,74.345,0.1015,813
54452,-1389884.060,-664401.38,285398.0,2448.040,63.42,65.250,71.660,0.1920,813


# fit an equivalent source model to the data

In [9]:
eqs = hm.EquivalentSources(
    damping=None,
    depth="default",
    block_size=2e3, # block reduce the data, use this to speed up testing but reduce value / remove for final run
)

eqs.fit((df.easting, df.northing, df.height), df.free_air_gravity_70s)

,damping,None
,points,None
,depth,'default'
,block_size,2000.0
,parallel,True
,dtype,'float64'


In [10]:
# Build the grid coordinates and set elevation as extra_coords (height for upward continuation)
grid_coords = vd.grid_coordinates(region=... , spacing=..., extra_coords=...)

# Predict the gravity disturbance on the grid
grav = eqs.grid(grid_coords, data_names = ['gravity_disturbance'])

TypeError: object of type 'ellipsis' has no len()

In [11]:
dampings = [0.01, 0.1, 1, 10,]
depths = [5e3, 10e3, 20e3, 50e3]

import itertools

parameter_sets = [
    dict(damping=combo[0], depth=combo[1])
    for combo in itertools.product(dampings, depths)
]
print("Number of combinations:", len(parameter_sets))
print("Combinations:", parameter_sets)

Number of combinations: 16
Combinations: [{'damping': 0.01, 'depth': 5000.0}, {'damping': 0.01, 'depth': 10000.0}, {'damping': 0.01, 'depth': 20000.0}, {'damping': 0.01, 'depth': 50000.0}, {'damping': 0.1, 'depth': 5000.0}, {'damping': 0.1, 'depth': 10000.0}, {'damping': 0.1, 'depth': 20000.0}, {'damping': 0.1, 'depth': 50000.0}, {'damping': 1, 'depth': 5000.0}, {'damping': 1, 'depth': 10000.0}, {'damping': 1, 'depth': 20000.0}, {'damping': 1, 'depth': 50000.0}, {'damping': 10, 'depth': 5000.0}, {'damping': 10, 'depth': 10000.0}, {'damping': 10, 'depth': 20000.0}, {'damping': 10, 'depth': 50000.0}]


In [13]:
eqs = hm.EquivalentSources()

scores = []
for params in parameter_sets:
    # set the damping and depth parameters
    eqs.set_params(**params)

    # fit the equivalent sources, and compute the cross-validation score
    score = np.mean(
        vd.cross_val_score(
            eqs,
            (df.easting, df.northing, df.height),
            df.free_air_gravity_70s,
        )
    )
    scores.append(score)
scores

MemoryError: Unable to allocate 14.1 GiB for an array with shape (43563, 43563) and data type float64

In [14]:
best = np.argmax(scores)
print("Best score:", scores[best])
print("Best parameters:", parameter_sets[best])

ValueError: attempt to get argmax of an empty sequence

In [16]:
eqs_best = hm.EquivalentSources(**parameter_sets[best]).fit(
     (df.easting, df.northing, df.height), df.free_air_gravity_70s,
)

# Predict the gravity disturbance on the grid
grav = eqs_best.grid(grid_coords, data_names = ['gravity_disturbance'])

NameError: name 'best' is not defined

In [17]:
bedmap_surface = fetch.bedmap3(
    layer="surface",
    region = ... ,
    reference="ellipsoid",
    # this is need to fill offshore nans with 0,
    # note, will fill with 0 geoidal height, then convert to ellipsoidal heights
    fill_nans=True,
)
bedmap2_surface.plot()

TypeError: 'ellipsis' object is not iterable

In [ ]:
density_contrast = ... - ...
zref = ...

# density contrasts are positive above the reference level, negative below
density_grid = xr.where(
    bedmap_surface >= zref,
    density_contrast,
    -density_contrast,
)

# create prism layer from ice surface and density grid
surface_prisms = hm.prism_layer(
    coordinates=(
        bedmap_surface.easting.to_numpy(),
        bedmap_surface.northing.to_numpy(),
    ),
    surface=bedmap_surface,
    reference= zref,
    properties={"density": density_grid},
)

In [18]:
# convert dataset to DataFrame for easier handling
grav_df = grav.to_dataframe().reset_index()

grav_df["surface_gravity"] = surface_prisms.prism_layer.gravity(
    coordinates=(
        grav_df.easting,
        grav_df.northing,
        grav_df.upward,
    ),
    field="g_z",
    progressbar=False,
)

# convert from dataframe back to xarray dataset
grav = grav_df.set_index(["northing", "easting"]).to_xarray()

NameError: name 'grav' is not defined